# UC-WP-2 — Content-Hash Idempotent Update

**Persona:** ETL Pipeline Engineer

**Goal:** Demonstrate content-hash gating for idempotent updates:

- `PUT` unchanged item body → content-hash matcher fires → `REFUSE_RETURN` (no DB write,
  existing item returned as `200`).
- `PUT` modified item body → content hash differs → upsert executes, updated properties
  reflected in the response.

This pattern is the correct idempotency primitive for pipelines that re-run on the same
source data but must propagate genuine updates without duplicating records.

**Use case:** UC-WP-2 — daily CHIRPS re-processing with conditional writes

**Shipped:** commit `18190f7` — 2026-04-15

**Prerequisites:**
- `content_hash` matcher enabled in the `IdentityMatcher` chain **after** `external_id`
  (see warning cell below)
- Item already pre-ingested or created in Step 0 below
- Env vars: `DYNASTORE_BASE_URL`, `DYNASTORE_TOKEN`, `CATALOG_ID`, `COLLECTION_ID`

In [ ]:
import hashlib
import json
import os
import uuid

import httpx
from dotenv import load_dotenv

load_dotenv()

BASE_URL = os.environ["DYNASTORE_BASE_URL"]
TOKEN = os.environ["DYNASTORE_TOKEN"]
CATALOG_ID = os.environ["CATALOG_ID"]
COLLECTION_ID = os.environ.get("COLLECTION_ID", "chirps_pentadal")

ITEMS_URL = f"/stac/catalogs/{CATALOG_ID}/collections/{COLLECTION_ID}/items"

headers = {
    "Authorization": f"Bearer {TOKEN}",
    "Content-Type": "application/json",
    "Accept": "application/json",
}
client = httpx.Client(base_url=BASE_URL, headers=headers, timeout=30.0)

RUN_ID = uuid.uuid4().hex[:8]
ITEM_ID = f"chirps-hash-test-{RUN_ID}"

print(f"BASE_URL     : {BASE_URL}")
print(f"CATALOG_ID   : {CATALOG_ID}")
print(f"COLLECTION_ID: {COLLECTION_ID}")
print(f"ITEM_ID      : {ITEM_ID}")


def canonical_hash(body: dict) -> str:
    """Compute a SHA-256 hash of a JSON body using canonical (sort_keys=True) serialisation."""
    serialised = json.dumps(body, sort_keys=True, separators=(",", ":"))
    return hashlib.sha256(serialised.encode("utf-8")).hexdigest()


print(f"canonical_hash helper ready")

In [ ]:
# Step 0 — Create the initial item (prerequisite)
initial_item = {
    "type": "Feature",
    "stac_version": "1.0.0",
    "id": ITEM_ID,
    "collection": COLLECTION_ID,
    "geometry": {
        "type": "Polygon",
        "coordinates": [[
            [35.0, 5.0],
            [45.0, 5.0],
            [45.0, 13.0],
            [35.0, 13.0],
            [35.0, 5.0],
        ]],
    },
    "bbox": [35.0, 5.0, 45.0, 13.0],
    "properties": {
        "datetime": "2024-02-05T00:00:00Z",
        "external_id": f"chirps-2024-02-P1-{RUN_ID}",
        "eo:cloud_cover": 5.0,
        "title": "CHIRPS Pentadal Feb 2024 P1",
    },
    "links": [],
    "assets": {},
}

resp = client.post(ITEMS_URL, content=json.dumps(initial_item))
print(resp.status_code, resp.text[:300])
assert resp.status_code == 201, f"Expected 201 on initial ingest, got {resp.status_code}: {resp.text}"

created = resp.json()
item_id = created["id"]
print(f"Initial item created: id={item_id}")

original_hash = canonical_hash(initial_item)
print(f"Original content hash: {original_hash}")

## Step 1 — PUT unchanged item (content-hash match → no write)

Submit the identical payload via `PUT`. The platform computes the content hash, finds a
match in the stored hash, and returns the existing item with `200`. No database write
is performed — the response is served directly from the match result.

In [ ]:
resp = client.put(f"{ITEMS_URL}/{item_id}", content=json.dumps(initial_item))
print(f"Status: {resp.status_code}")
print(f"Body  : {resp.text[:400]}")

# Content-hash match must result in REFUSE_RETURN (200) — not a re-write (200/201 from upsert)
assert resp.status_code == 200, (
    f"Expected 200 (content-hash REFUSE_RETURN) for unchanged PUT, "
    f"got {resp.status_code}: {resp.text}"
)

returned = resp.json()
assert "id" in returned, "Response must contain 'id'"
assert returned["id"] == item_id, (
    f"Returned id {returned['id']!r} does not match original {item_id!r}"
)
print(f"Confirmed: existing item returned (id={returned['id']}) — no write performed.")

In [ ]:
print("content hash matched — no write performed")

## Step 2 — PUT modified item (content changed → upsert)

Change one property (`eo:cloud_cover`) to produce a different content hash. The platform
detects the hash mismatch and executes the upsert. The updated value must be reflected
in the response body.

In [ ]:
import copy

modified_item = copy.deepcopy(initial_item)
modified_item["properties"]["eo:cloud_cover"] = 42.0  # changed from 5.0
modified_item["properties"]["title"] = "CHIRPS Pentadal Feb 2024 P1 (revised)"

resp = client.put(f"{ITEMS_URL}/{item_id}", content=json.dumps(modified_item))
print(f"Status: {resp.status_code}")
print(f"Body  : {resp.text[:400]}")

assert resp.status_code == 200, (
    f"Expected 200 (upsert executed) for modified PUT, got {resp.status_code}: {resp.text}"
)

updated = resp.json()
updated_cloud_cover = updated.get("properties", {}).get("eo:cloud_cover")
assert updated_cloud_cover == 42.0, (
    f"Expected eo:cloud_cover=42.0 in updated item, got {updated_cloud_cover!r}"
)
print(f"Updated eo:cloud_cover confirmed: {updated_cloud_cover}")

In [ ]:
# Verify hash difference between original and modified body
modified_hash = canonical_hash(modified_item)
print(f"Original hash: {original_hash}")
print(f"Modified hash: {modified_hash}")

assert original_hash != modified_hash, (
    "Original and modified item bodies must produce different content hashes"
)
print("Content hashes differ — upsert was correctly triggered.")

## Edge cases

### Canonical JSON ordering — sort_keys=True is required

Content-hash comparison is sensitive to key ordering. Two semantically identical JSON
objects serialised with different key orderings produce different byte sequences and
therefore different hashes. Always use `sort_keys=True` (or an equivalent canonical
serialiser) before hashing.

In [ ]:
sample = {"z": 1, "a": 2, "m": 3}

# Bad serialisation: Python dict insertion order is preserved but not canonical
bad_serial = json.dumps(sample, separators=(",", ":"))
bad_hash = hashlib.sha256(bad_serial.encode()).hexdigest()

# Good serialisation: sort_keys=True gives a deterministic canonical form
good_serial = json.dumps(sample, sort_keys=True, separators=(",", ":"))
good_hash = hashlib.sha256(good_serial.encode()).hexdigest()

print(f"Bad  serialisation: {bad_serial}")
print(f"Bad  hash         : {bad_hash}")
print()
print(f"Good serialisation: {good_serial}")
print(f"Good hash         : {good_hash}")

# Demonstrate the problem: shuffled key order → different hash without sort_keys
shuffled = {"a": 2, "m": 3, "z": 1}  # same values, different key insertion order
shuffled_bad = json.dumps(shuffled, separators=(",", ":"))
shuffled_hash = hashlib.sha256(shuffled_bad.encode()).hexdigest()
print()
print(f"Shuffled (bad) serialisation: {shuffled_bad}")
print(f"Shuffled hash (bad)         : {shuffled_hash}")
print(f"Hashes differ without sort_keys: {bad_hash != shuffled_hash}")

# With sort_keys both orderings produce the same hash
shuffled_good = json.dumps(shuffled, sort_keys=True, separators=(",", ":"))
shuffled_good_hash = hashlib.sha256(shuffled_good.encode()).hexdigest()
assert good_hash == shuffled_good_hash, (
    "sort_keys=True must produce identical hashes regardless of insertion order"
)
print()
print(f"With sort_keys=True hashes are identical: {good_hash == shuffled_good_hash}")
print("Always use canonical_hash() — never raw json.dumps() — before comparing.")

> **Warning:** Place `content_hash` **after** `external_id` in the `IdentityMatcher`
> chain, never before.
>
> If `content_hash` is evaluated first, two items with different `external_id` values but
> accidentally identical content (e.g. empty geometries or template defaults) would be
> collapsed into one record, silently discarding the second item's `external_id`. The
> correct chain order is: `external_id` → `geohash` → `content_hash`.

## Teardown

In [ ]:
resp = client.delete(f"{ITEMS_URL}/{item_id}")
print(f"DELETE {item_id}: {resp.status_code}")
assert resp.status_code in (204, 404), (
    f"Expected 204 or 404 on teardown delete, got {resp.status_code}: {resp.text}"
)
print("Teardown complete.")
client.close()